# Semantic Representation Pengaduan Surabaya 

In [ ]:
from pathlib import Path
import random, time, platform
import numpy as np
import pandas as pd


def find_file(patterns, search_dirs):
    for d in search_dirs:
        for p in patterns:
            hits = sorted(Path(d).glob(p))
            if hits:
                return hits[0]
    return None


BASE_DIR = Path.cwd()
SEARCH_DIRS = [
    BASE_DIR, BASE_DIR / "cleaning_output", BASE_DIR.parent / "cleaning_output",
    Path("/content"), Path("/content/drive/MyDrive"),
]
DATA_PATH = find_file(["data_cleaned.csv", "data_cleaned*.csv"], SEARCH_DIRS)
assert DATA_PATH is not None, "data_cleaned.csv tidak ditemukan. Unggah / letakkan file dari Notebook 02 di folder kerja."
OUTPUT_DIR = BASE_DIR / "semantic_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ROWS = 33880        
MODEL_NAME = "indobenchmark/indobert-base-p1"
SEED = 42
MAX_LENGTH = 256              
CHUNK_SIZE = 2000              
LOW_VOLUME_THRESHOLD = 10

REPRESENTASI_UTAMA = "mean_centered"

np.random.seed(SEED)
random.seed(SEED)

print("DATA_PATH :", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL     :", MODEL_NAME)

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

required_cols = [
    "no_tiket", "tahun", "teks_embedding", "kelurahan", "kecamatan",
    "lokasi_key", "lat_kelurahan", "lon_kelurahan", "kelompok_topik",
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Kolom wajib tidak ditemukan: {missing}"
assert len(df) == EXPECTED_ROWS, f"Jumlah baris {len(df)} != {EXPECTED_ROWS}: pastikan memakai data_cleaned.csv final dari Notebook 02."
assert df["no_tiket"].is_unique, "no_tiket tidak unik"

print("Shape:", df.shape)
print("Unique lokasi_key:", df["lokasi_key"].nunique())
print(df["tahun"].value_counts().sort_index())

In [ ]:
text_col = "teks_embedding"

is_missing = df[text_col].isna()
is_empty = df[text_col].fillna("").astype(str).str.strip().eq("")

df_embed = (
    df.loc[~is_missing & ~is_empty]
      .copy()
      .reset_index(drop=False)
      .rename(columns={"index": "source_row"})
)

print("Missing text:", int(is_missing.sum()))
print("Empty text:", int(is_empty.sum()))
print("Rows ready for embedding:", len(df_embed))
assert len(df_embed) > 0

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64 if device.type == "cuda" else 16
print("Device:", device, "| batch size:", BATCH_SIZE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

hidden_size = int(model.config.hidden_size)
print("Hidden size:", hidden_size)


@torch.no_grad()
def embed_batch(batch_texts):
    """Mengembalikan (cls, mean) untuk satu batch. Mean mengabaikan token padding."""
    enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc).last_hidden_state                       # (B, T, H)
    cls = out[:, 0, :]
    mask = enc["attention_mask"].unsqueeze(-1).to(out.dtype)
    mean = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return cls.float().cpu().numpy(), mean.float().cpu().numpy()

## Audit panjang teks

Berapa tiket yang melebihi `MAX_LENGTH` token dan akan terpotong. Tiket tidak dibuang; angka ini dicatat sebagai keterbatasan.

In [ ]:
texts = df_embed[text_col].astype(str).tolist()

n_tokens = []
for start in range(0, len(texts), 2048):
    ids = tokenizer(texts[start:start + 2048], add_special_tokens=True, truncation=False, padding=False)["input_ids"]
    n_tokens.extend(len(x) for x in ids)
n_tokens = np.asarray(n_tokens)

n_truncated = int((n_tokens > MAX_LENGTH).sum())
print(f"Token per tiket: median={int(np.median(n_tokens))}, p95={int(np.percentile(n_tokens, 95))}, max={int(n_tokens.max())}")
print(f"Tiket yang terpotong pada MAX_LENGTH={MAX_LENGTH}: {n_truncated} ({n_truncated / len(n_tokens):.2%})")
print(f"Tiket > 512 token (di atas batas model): {int((n_tokens > 512).sum())}")
df_embed["n_token"] = n_tokens

## Embedding per pengaduan

Tiket diurutkan menurut panjang token supaya padding minimal, diproses per potongan (`CHUNK_SIZE`), dan tiap potongan disimpan. Jika sesi terputus, jalankan ulang sel ini: potongan yang sudah ada dilewati.

In [ ]:
import hashlib

tag = f"{MODEL_NAME.split('/')[-1]}_L{MAX_LENGTH}"
chunks_dir = OUTPUT_DIR / "chunks" / tag
chunks_dir.mkdir(parents=True, exist_ok=True)


def chunk_signature(idx):
    """Hash isi teks + konfigurasi untuk satu potongan; cache dipakai hanya bila teks dan konfigurasi identik."""
    h = hashlib.sha256(f"{MODEL_NAME}|{MAX_LENGTH}|{len(idx)}".encode("utf-8"))
    for i in idx:
        h.update(texts[i].encode("utf-8"))
        h.update(b"\x00")
    return h.hexdigest()


n = len(texts)
order = np.argsort(n_tokens, kind="stable")
cls_all = np.zeros((n, hidden_size), dtype=np.float32)
mean_all = np.zeros((n, hidden_size), dtype=np.float32)

t0 = time.time()
n_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
n_reuse = 0
for ci, c_start in enumerate(range(0, n, CHUNK_SIZE), start=1):
    ids = order[c_start:c_start + CHUNK_SIZE]
    sig = chunk_signature(ids)
    f = chunks_dir / f"chunk_{c_start:06d}.npz"
    if f.exists():
        z = np.load(f)
        ok = ("sig" in z.files and str(z["sig"]) == sig
              and z["ids"].shape == ids.shape and (z["ids"] == ids).all())
        if ok:
            cls_all[ids], mean_all[ids] = z["cls"], z["mean"]
            n_reuse += 1
            continue
    cls_parts, mean_parts = [], []
    for b in tqdm(range(0, len(ids), BATCH_SIZE), desc=f"potongan {ci}/{n_chunks}", leave=False):
        bi = ids[b:b + BATCH_SIZE]
        c, m = embed_batch([texts[i] for i in bi])
        cls_parts.append(c); mean_parts.append(m)
    c, m = np.vstack(cls_parts), np.vstack(mean_parts)
    np.savez(f, ids=ids, cls=c, mean=m, sig=np.array(sig))
    cls_all[ids], mean_all[ids] = c, m

elapsed_sec = round(time.time() - t0, 1)
print(f"Potongan dipakai ulang dari cache: {n_reuse}/{n_chunks} | dihitung ulang: {n_chunks - n_reuse}")
print("Waktu sesi ini (detik):", elapsed_sec)
print("Embedding shape:", cls_all.shape, mean_all.shape)

assert cls_all.shape == (len(df_embed), hidden_size) and mean_all.shape == cls_all.shape
assert np.isfinite(cls_all).all() and np.isfinite(mean_all).all()
assert (np.abs(cls_all).sum(axis=1) > 0).all() and (np.abs(mean_all).sum(axis=1) > 0).all(), "Ada baris embedding kosong"

In [ ]:
def l2(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)

VARIAN = {
    "cls_raw": l2(cls_all),
    "cls_centered": l2(cls_all - cls_all.mean(axis=0, keepdims=True)),
    "mean_raw": l2(mean_all),
    "mean_centered": l2(mean_all - mean_all.mean(axis=0, keepdims=True)),
}
assert REPRESENTASI_UTAMA in VARIAN, f"REPRESENTASI_UTAMA harus salah satu dari {list(VARIAN)}"
for name, V in VARIAN.items():
    assert np.allclose(np.linalg.norm(V, axis=1), 1.0, atol=1e-4), name

meta_cols = [
    "source_row", "no_tiket", "tahun", "kelurahan", "kecamatan",
    "lokasi_key", "lat_kelurahan", "lon_kelurahan", "kelompok_topik", "n_token",
]
embedding_meta = df_embed[meta_cols].copy().reset_index(drop=True)
embedding_meta["embedding_index"] = np.arange(len(embedding_meta))

np.save(OUTPUT_DIR / "complaint_embeddings_raw_cls.npy", cls_all)
np.save(OUTPUT_DIR / "complaint_embeddings_raw_mean.npy", mean_all)
embedding_meta.to_csv(OUTPUT_DIR / "complaint_embedding_metadata.csv", index=False, encoding="utf-8-sig")
print("Complaint embeddings saved (raw cls + raw mean; varian lain dapat dibuat ulang dengan l2 / centering).")

## Mean pooling per kelurahan x tahun

Node dibentuk langsung dalam urutan akhir (tahun, kecamatan, kelurahan), sehingga `node_id`, tabel node, dan baris vektor selalu sejajar.

In [ ]:
group_cols = ["lokasi_key", "kelurahan", "kecamatan", "tahun"]
idx_by_key = embedding_meta.groupby(group_cols).indices          # (lokasi_key, kelurahan, kecamatan, tahun) -> posisi baris
keys_sorted = sorted(idx_by_key, key=lambda k: (int(k[3]), k[2], k[1], k[0]))

records = []
for k in keys_sorted:
    first = embedding_meta.iloc[idx_by_key[k][0]]
    records.append({
        "lokasi_key": k[0], "kelurahan": k[1], "kecamatan": k[2], "tahun": int(k[3]),
        "n_pengaduan": int(len(idx_by_key[k])),
        "lat_kelurahan": first["lat_kelurahan"], "lon_kelurahan": first["lon_kelurahan"],
    })
node_table = pd.DataFrame(records)
node_table["node_id"] = np.arange(len(node_table))

node_vectors = {
    name: l2(np.vstack([V[idx_by_key[k]].mean(axis=0) for k in keys_sorted])).astype(np.float32)
    for name, V in VARIAN.items()
}
print(node_table.shape, {k: v.shape for k, v in node_vectors.items()})

In [ ]:
# Validasi target 459 node
assert node_table["lokasi_key"].nunique() == 153
assert set(node_table["tahun"].unique()) == {2023, 2024, 2025}
assert len(node_table) == 459
assert node_table["node_id"].is_unique
assert node_table.groupby("lokasi_key")["tahun"].nunique().eq(3).all()
assert node_table.groupby("tahun")["lokasi_key"].nunique().eq(153).all()
assert int(node_table["n_pengaduan"].sum()) == len(df_embed), "Jumlah pengaduan per node tidak sama dengan total"
for name, X in node_vectors.items():
    assert X.shape == (459, hidden_size) and np.isfinite(X).all(), name

# Uji silang: vektor node harus sama dengan rata-rata embedding pengaduan miliknya sendiri
rng = np.random.default_rng(SEED)
for i in rng.choice(len(node_table), size=8, replace=False):
    r = node_table.iloc[i]
    pos = embedding_meta.index[(embedding_meta["lokasi_key"] == r["lokasi_key"]) & (embedding_meta["tahun"] == r["tahun"])].to_numpy()
    assert len(pos) == r["n_pengaduan"]
    for name, V in VARIAN.items():
        v = V[pos].mean(axis=0); v = v / np.linalg.norm(v)
        assert np.allclose(v, node_vectors[name][i], atol=1e-5), (name, i)

print("PASS: 153 kelurahan x 3 tahun = 459 node; vektor node sejajar dengan node_table")

## Diagnostik cosine antar node

Seberapa berbeda satu node dari node lain pada tiap representasi. Jika semua pasangan node memiliki cosine yang hampir sama (rata-rata sangat tinggi, sebaran sempit), bobot sisi semantik pada graf nyaris seragam dan community detection sulit membedakan komunitas.

`gap_kelurahan_sama` = rata-rata cosine antar tahun untuk kelurahan yang sama dikurangi rata-rata semua pasangan. Semakin besar, semakin kuat representasi menangkap ciri khas kelurahan

In [ ]:
iu = np.triu_indices(len(node_table), k=1)

pairs_same = []
for _, g in node_table.groupby("lokasi_key"):
    ids = g["node_id"].to_numpy()
    for a in range(len(ids)):
        for b in range(a + 1, len(ids)):
            pairs_same.append((ids[a], ids[b]))
pairs_same = np.asarray(pairs_same)

rows = []
for name, X in node_vectors.items():
    S = X @ X.T
    c = S[iu]
    same = (X[pairs_same[:, 0]] * X[pairs_same[:, 1]]).sum(axis=1)
    rows.append({
        "representasi": name,
        "cos_mean": c.mean(), "cos_std": c.std(),
        "cos_p05": np.percentile(c, 5), "cos_p50": np.percentile(c, 50), "cos_p95": np.percentile(c, 95),
        "cos_min": c.min(), "cos_max": c.max(),
        "cos_kelurahan_sama_beda_tahun": same.mean(),
        "gap_kelurahan_sama": same.mean() - c.mean(),
    })
diagnostik = pd.DataFrame(rows).round(4)
display(diagnostik)
diagnostik.to_csv(OUTPUT_DIR / "embedding_diagnostics.csv", index=False, encoding="utf-8-sig")

In [ ]:
low_volume_nodes = (
    node_table[node_table["n_pengaduan"] < LOW_VOLUME_THRESHOLD]
    .sort_values(["tahun", "n_pengaduan", "kelurahan"])
    .reset_index(drop=True)
)
print(f"Node dengan < {LOW_VOLUME_THRESHOLD} pengaduan: {len(low_volume_nodes)}")
display(low_volume_nodes)
low_volume_nodes.to_csv(OUTPUT_DIR / "low_volume_nodes.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Simpan semua varian node + varian utama
for name, X in node_vectors.items():
    np.save(OUTPUT_DIR / f"node_embeddings_{name}.npy", X)
np.save(OUTPUT_DIR / "node_embeddings_kelurahan_tahun.npy", node_vectors[REPRESENTASI_UTAMA])
node_table.to_csv(OUTPUT_DIR / "node_table_kelurahan_tahun.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame([{
    "clean_rows": len(df),
    "embedded_rows": len(df_embed),
    "embedding_dim": hidden_size,
    "kelurahan": node_table["lokasi_key"].nunique(),
    "years": "2023,2024,2025",
    "nodes": len(node_table),
    "low_volume_nodes_lt_threshold": int((node_table["n_pengaduan"] < LOW_VOLUME_THRESHOLD).sum()),
    "low_volume_threshold": LOW_VOLUME_THRESHOLD,
    "representasi_utama": REPRESENTASI_UTAMA,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "n_truncated": n_truncated,
    "pct_truncated": round(n_truncated / len(df_embed), 4),
    "batch_size": BATCH_SIZE,
    "device": str(device),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "python": platform.python_version(),
    "seed": SEED,
    "elapsed_sec_sesi_ini": elapsed_sec,
}])
summary.to_csv(OUTPUT_DIR / "semantic_embedding_summary.csv", index=False, encoding="utf-8-sig")
display(summary.T)
print("STATUS: PASS")

In [ ]:
tp = (df.groupby(["lokasi_key","tahun"])["kelompok_topik"]
        .value_counts(normalize=True).unstack(fill_value=0))
tp = tp.loc[[(r.lokasi_key, r.tahun) for r in node_table.itertuples()]].to_numpy()
tp_n = tp / np.linalg.norm(tp, axis=1, keepdims=True)
S_topik = (tp_n @ tp_n.T)[iu]

from scipy.stats import spearmanr
for name, X in node_vectors.items():
    print(name, round(spearmanr((X @ X.T)[iu], S_topik)[0], 3))